# Exercises XP Ninja: W4_D4

## How to run in Colab
1. I open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** and choose this file.
2. I run the first cell. If Evidently is not installed, it installs **evidently==0.6.7** and restarts the session. Then I run all the cells again from the top.
3. No zip upload is needed here: the notebook downloads the Drug Review dataset (Drugs.com) directly from the UCI website.
4. The translation step first uses Google Translate. If Google blocks the requests (rate limit or captcha), the notebook falls back to a local Hugging Face English→French model (`Helsinki-NLP/opus-mt-en-fr`). The first time, it downloads the model (about 300 MB), and translating on CPU takes a few minutes.
5. I don't run any extra `pip install` / upgrade of Evidently: the newest version (0.7+) removed `ColumnMapping` and the `evidently.report` API used in this tutorial.

In [1]:
import os
try:
    import evidently
    print(f"Evidently {evidently.__version__} is already available.")
except ImportError:
    print("Evidently not found. Performing clean installation...")
    # I pin 0.6.7 (not 0.7.21): the later cells use evidently.report / ColumnMapping / evidently.features,
    # which only exist in this form before 0.7. The markdown below also says we pin 0.6.7.
    !pip install --no-cache-dir evidently==0.6.7
    print("Restarting session to apply changes...")
    os._exit(0)

import importlib
importlib.invalidate_caches()

try:
    from evidently.report import Report
    from evidently.metrics import ClassificationQualityMetric, ColumnDriftMetric
    from evidently.metric_preset import DataDriftPreset
    from evidently import ColumnMapping
except ImportError:
    from evidently.legacy.report import Report
    from evidently.legacy.metrics import ClassificationQualityMetric, ColumnDriftMetric
    from evidently.legacy.metric_preset import DataDriftPreset
    from evidently.legacy.pipeline.column_mapping import ColumnMapping

print(f"Evidently {evidently.__version__} successfully imported! \u2705")

Evidently 0.6.7 is already available.
Evidently 0.6.7 successfully imported! ✅


In [2]:
import importlib
importlib.invalidate_caches()

try:
    from evidently.report import Report
    from evidently.metrics import ClassificationQualityMetric, ColumnDriftMetric
    from evidently.metric_preset import DataDriftPreset
    from evidently import ColumnMapping
    from evidently.metrics import TextDescriptorsDriftMetric
except ImportError:
    from evidently.legacy.report import Report
    from evidently.legacy.metrics import ClassificationQualityMetric, ColumnDriftMetric, TextDescriptorsDriftMetric
    from evidently.legacy.metric_preset import DataDriftPreset
    from evidently.legacy.pipeline.column_mapping import ColumnMapping

print("Imports Evidently OK \u2705")

Imports Evidently OK ✅


In [3]:
# Restoration of the environment after kernel restart
import pandas as pd
from sklearn.model_selection import train_test_split

if 'raw_data' in locals():
    # 1. Filter and Prepare
    filtered_data = raw_data[(raw_data['condition'] == 'Pain') & (raw_data['rating'].isin([1, 10]))].copy()
    filtered_data['is_positive'] = filtered_data['rating'].apply(lambda x: 1 if x == 10 else 0)
    filtered_data.drop('rating', inplace=True, axis=1)

    # 2. Split
    X_train, X_test, y_train, y_test = train_test_split(
        filtered_data['review'], filtered_data['is_positive'], test_size=0.4, random_state=42, shuffle=True
    )
    reference = pd.DataFrame({'review': X_train, 'is_positive': y_train})
    valid = pd.DataFrame({'review': X_test, 'is_positive': y_test})

    # 3. Predict (Ensure pipeline is fitted in cell Dr9Qlun_7LqR first)
    if 'pipeline' in locals():
        reference['predict_proba'] = pipeline.predict_proba(reference['review'].values)[:,1]
        valid['predict_proba'] = pipeline.predict_proba(valid['review'].values)[:,1]

        # 4. Create the disturbed dataset
        valid_disturbed = valid[['review', 'is_positive', 'predict_proba']].copy()
        print("Environnement de données restauré ✅. Vous pouvez lancer les rapports.")
    else:
        print("Pipeline non trouvé. Veuillez exécuter la cellule d'entraînement du modèle (Dr9Qlun_7LqR).")
else:
    print("Données non trouvées. Veuillez exécuter la cellule de chargement (wgt0CmMj1C3Z).")

Données non trouvées. Veuillez exécuter la cellule de chargement (wgt0CmMj1C3Z).


In [4]:
import requests
import zipfile
import io
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

# 1. Reload Data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
content = requests.get(url).content
with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("drugsComTest_raw.tsv"), sep='\t')
raw_data = raw_data[['drugName', 'condition', 'review', 'rating']]

# 2. Filter
filtered_data = raw_data[(raw_data['condition'] == 'Pain') & (raw_data['rating'].isin([1, 10]))].copy()
filtered_data['is_positive'] = filtered_data['rating'].apply(lambda x: 1 if x == 10 else 0)
filtered_data.drop('rating', inplace=True, axis=1)

# 3. Split
X_train, X_test, y_train, y_test = train_test_split(filtered_data['review'], filtered_data['is_positive'], test_size=0.4, random_state=42, shuffle=True)
reference = pd.DataFrame({'review': X_train, 'is_positive': y_train})
valid = pd.DataFrame({'review': X_test, 'is_positive': y_test})

# 4. Train Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, max_df=0.5, stop_words='english')),
    ('classifier', SGDClassifier(alpha=0.0001, max_iter=50, penalty='l1', loss='modified_huber', random_state=42))
])
pipeline.fit(X_train, y_train)

# 5. Predictions
reference['predict_proba'] = pipeline.predict_proba(reference['review'].values)[:,1]
valid['predict_proba'] = pipeline.predict_proba(valid['review'].values)[:,1]
valid_disturbed = valid[['review', 'is_positive', 'predict_proba']].copy()

print("Variables et modèle restaurés avec succès ! ✅")

Variables et modèle restaurés avec succès ! ✅


# Imports and support functions

In [5]:
import pandas as pd
import requests
import zipfile
import io
import random
from sklearn.model_selection import train_test_split
# Ensuring all necessary libraries are imported.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

In [7]:
# Title: Verify pinned versions
# Comments: After pinning, sklearn should be 1.4.2, numpy ~2.0.x, scipy 1.13.x.

import importlib

def v(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return "not installed"

print("scikit-learn:", v("sklearn"))
print("numpy       :", v("numpy"))
print("scipy       :", v("scipy"))

scikit-learn: 1.6.1
numpy       : 2.0.2
scipy       : 1.14.1


In [8]:
# 1. Install latest evidently and fix dependency conflicts
# I skip this upgrade on purpose: it installs the newest Evidently (0.7+), which removes the API this notebook uses.
# !pip install --no-cache-dir --upgrade evidently pydantic typing-extensions

# 2. Verify physical presence of the report module across all paths
import os
import site
import sys

found = False
for path in site.getsitepackages() + sys.path:
    report_dir = os.path.join(path, 'evidently', 'report')
    if os.path.exists(report_dir):
        print(f"✓ SUCCESS: 'report' module found at {report_dir}")
        found = True
        break

if not found:
    print("❌ 'report' is still missing. Attempting to locate the 'evidently' root...")
    import evidently
    print(os.listdir(os.path.dirname(evidently.__file__)))

✓ SUCCESS: 'report' module found at /workspace/w4d3/venv_ninja/lib/python3.12/site-packages/evidently/report


In [9]:
import sys
import importlib
import site

# Force refresh the module cache
importlib.invalidate_caches()
importlib.reload(site)

try:
    # Try modern path first, then fallback to legacy path found in the file system
    try:
        from evidently.report import Report
        from evidently.metric_preset import DataDriftPreset
    except ImportError:
        from evidently.legacy.report import Report
        from evidently.legacy.metric_preset import DataDriftPreset

    print("✓ SUCCESS: Evidently imports (via legacy path) are working!")
    print("You can now proceed with your analysis.")
except ImportError as e:
    print(f"❌ Final import check failed: {e}")

✓ SUCCESS: Evidently imports (via legacy path) are working!
You can now proceed with your analysis.


In [10]:
# Title: Optional cleanup of unused heavy libs
# Comments: Only run if you want to reduce resolver noise; not needed for this notebook.
%pip uninstall -y tsfresh cuml-cu12 umap-learn

Note: you may need to restart the kernel to use updated packages.


We skip !pip install googletrans==4.0.0-rc1 because it pins an old httpx (0.13.3) and breaks the environment. We use deep_translator instead.

In [11]:
# !pip install googletrans==4.0.0-rc1

We skip installing Evidently from main (!pip install git+https://github.com/evidentlyai/evidently.git@main) because the 0.7+ API removed ColumnMapping used in this tutorial. We pin evidently==0.6.7 which matches the notebook code.

In [12]:
# !pip install git+https://github.com/evidentlyai/evidently.git@main

If a cell tries to auto-install Evidently from GitHub in a try/except, we do not run it; we keep the pinned version for consistency.

Imports adjusted to be robust:

In [13]:
# Tentative d'importation d'evidently
# Note : Correction de l'indentation de l'except et ajout de commentaires
try:
    import evidently
    print("Evidently est déjà installé.")
except ImportError:
    # On ne réinstalle pas depuis master car nous avons déjà fixé la version 0.7+
    print("Evidently n'est pas trouvé, installation manuelle ignorée pour préserver la stabilité.")

Evidently est déjà installé.


In [14]:
# Title: Stable Evidently imports (no re-install)
# Comments: We skip pip installs to avoid conflicts and use the legacy API compatible with evidently==0.6.7.

# SKIP problematic installs:
# !pip install googletrans==4.0.0-rc1
# !pip install git+https://github.com/evidentlyai/evidently.git@main
# try:
#     import evidently
# except:
#     !pip install git+https://github.com/evidentlyai/evidently.git

# Keep and adapt imports:
import evidently
print("Evidently version:", evidently.__version__)

# ColumnMapping import (robust for 0.6.7 and older variants)
try:
    from evidently import ColumnMapping
except ImportError:
    from evidently.pipeline.column_mapping import ColumnMapping

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset, ClassificationPreset
from evidently.metrics import ClassificationQualityMetric, TextDescriptorsDriftMetric, ColumnDriftMetric

print("Evidently imports OK ✅")

Evidently version: 0.6.7
Evidently imports OK ✅


# Load data

We will work with a dataset that contains reviews and ratings for different drugs.

In [15]:
import requests
import zipfile
import io
import pandas as pd

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
content = requests.get(url).content
with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("drugsComTest_raw.tsv"), sep='\t')

raw_data = raw_data[['drugName', 'condition', 'review', 'rating']]
print("Data loaded successfully. Shape:", raw_data.shape)

Data loaded successfully. Shape: (53766, 4)


Data source: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

Citation:
Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder. 2018. Aspect-Based Sentiment Analysis of Drug Reviews Applying Cross-Domain and Cross-Data Learning. In Proceedings of the 2018 International Conference on Digital Health (DH '18). ACM, New York, NY, USA, 121-125. DOI: [Web Link]

In [16]:
print(raw_data.head(5))


          drugName                     condition  \
0      Mirtazapine                    Depression   
1       Mesalamine  Crohn's Disease, Maintenance   
2          Bactrim       Urinary Tract Infection   
3         Contrave                   Weight Loss   
4  Cyclafem 1 / 35                 Birth Control   

                                              review  rating  
0  "I&#039;ve tried a few antidepressants over th...    10.0  
1  "My son has Crohn&#039;s disease and has done ...     8.0  
2                      "Quick reduction of symptoms"     9.0  
3  "Contrave combines drugs that were used for al...     9.0  
4  "I have been on this birth control for one cyc...     9.0  


# Training and testing the model

Suppose we want to build a model to distinguish between reviews with rating 1 (negative review) and 10 (positive review). Let's also assume that we only have access to reviews on pain medications.

1. Filter Reviews by Condition and Rating: Start by selecting only the reviews related to the condition 'Pain' and with ratings of either 1 or 10. This step is crucial for focusing on the specific subset of interest.

2. Create a Target Variable: Transform the rating column into a binary target variable named is_positive. In this variable, a rating of 1 should be represented as 0 (indicating a negative review), and a rating of 10 should be represented as 1 (indicating a positive review). This transformation is necessary for binary classification tasks.

3. Drop the Original Rating Column: Once you have created the is_positive column, the original rating column is redundant. Remove it from your dataset to clean up your data.

4. Preview the Dataset: Finally, display the first few rows of your dataset to ensure everything looks correct.

Hint: Use the .loc[] method for filtering rows based on conditions, the .isin() method to filter the dataframe by specific rating values, and the .apply() function to create a new column based on conditional logic. Remember to use inplace=True when dropping a column to modify the original DataFrame.

In [17]:
# Filter the data
filtered_data = raw_data[(raw_data['condition'] == 'Pain') & (raw_data['rating'].isin([1, 10]))].copy()

# Create the binary target variable
filtered_data['is_positive'] = filtered_data['rating'].apply(lambda x: 1 if x == 10 else 0)

# Drop the original 'rating' column
filtered_data.drop('rating', inplace=True, axis=1)

# Preview the dataset
print(f"Filtered data shape: {filtered_data.shape}")
print(filtered_data.head())

Filtered data shape: (926, 4)
                    drugName condition  \
77                    Ultram      Pain   
120                Oxycodone      Pain   
140                 Tramadol      Pain   
141  Tylenol with Codeine #3      Pain   
165                  Demerol      Pain   

                                                review  is_positive  
77   "I have severe scar tissue and adhesions from ...            1  
120  "In 2000 I had 4 surgeries to repair an anal f...            1  
140  "This medication does not relieve eve the slig...            0  
141  "After I was discharged from intensive care, I...            0  
165  "I was given Demorol for pain in the ER about ...            1  


We split the data into "reference" and "valid" datasets. Reference dataset is used for training while 40% of the data is held out for model validation

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(filtered_data['review'], filtered_data['is_positive'],
                                                    test_size=0.4, random_state=42, shuffle=True)

# Create DataFrames for the reference (training) and validation (testing) datasets
reference = pd.DataFrame({'review': X_train, 'is_positive': y_train})
valid = pd.DataFrame({'review': X_test, 'is_positive': y_test})

Train a model with TF-IDF vectorization and linear classifier on top

1. Set Up the Pipeline: Create a pipeline that includes two main steps - vectorization and classification. For vectorization, use the TfidfVectorizer to transform the review texts into TF-IDF features. For classification, use a linear classifier, specifically SGDClassifier, to predict whether a review is negative or positive based on the TF-IDF features.

2. Configure the TfidfVectorizer: In the vectorization step, set sublinear_tf=True to use a logarithmic form for frequency, max_df=0.5 to ignore terms that appear in more than 50% of the documents, and specify stop_words="english" to remove common English stop words that are of little value in the analysis.

3. Configure the SGDClassifier: For the classifier, use the following settings: alpha=0.0001 for regularization strength, max_iter=50 for the number of passes over the training data (epochs), penalty='l1' for the L1 regularization, loss='modified_huber' for the loss function, and set random_state=42 for reproducibility.

4. Train the Model: Fit your pipeline to the data. You will need two arrays: one representing the texts of the reviews and the other representing the binary target variable (indicating negative or positive reviews).

5. Verify the Model: After training, use the .predict() method on your pipeline to make predictions on new data and evaluate the performance of your model.

Hint: The Pipeline function from sklearn.pipeline allows you to chain multiple steps together, applying them sequentially. This is ideal for combining a vectorization step with a classification step into a single process.

In [20]:
# Set up the pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, max_df=0.5, stop_words='english')),
    ('classifier', SGDClassifier(alpha=0.0001, max_iter=50, penalty='l1', loss='modified_huber', random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.5, stop_words='english',
                                 sublinear_tf=True)),
                ('classifier',
                 SGDClassifier(loss='modified_huber', max_iter=50, penalty='l1',
                               random_state=42))])

Calculate model predictions for training and validation datasets. Our model predicts the probability of a review being positive

In [21]:
reference['predict_proba'] = pipeline.predict_proba(reference['review'].values)[:,1]
valid['predict_proba'] = pipeline.predict_proba(valid['review'].values)[:,1]

In [22]:
# set up column mapping
column_mapping = ColumnMapping()

column_mapping.target = 'is_positive'
column_mapping.prediction = 'predict_proba'
column_mapping.text_features = ['review']

# list features so text field is not treated as a regular feature
column_mapping.numerical_features = []
column_mapping.categorical_features = []

Model accuracy on validation dataset is a bit higher than 0.8. This is the level of performance we can expect on similar new data

In [23]:
performance_report = Report(metrics=[
    ClassificationQualityMetric()
])

performance_report.run(reference_data=reference, current_data=valid,
                        column_mapping=column_mapping)
performance_report

I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


# Data drift due to "technical issues"

Imagine that after deploying the model something changes at the data collecting stage. Could be that on-line data preprocessing differs from historical one or that some odd update simply breaks data cleaning steps.

Let's see what happens if we either inject some random html tags in the review text or translate the review to French.

In [24]:
!pip install deep_translator


In [25]:
from deep_translator import GoogleTranslator
import random

def translate_str(s):
    translator = GoogleTranslator(source='auto', target='fr')
    return translator.translate(s)

random_html_tags = (
    '<body>, </body>', '<html><body>', '</body></html>', '<h1>', '</h1>',
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>'
)

def inject_random_html_tags(s):
    num_tags = 25
    for i in range(num_tags):
        random.seed(i)
        pos = random.choice(range(len(s)))
        s = s[:pos] + random.choice(random_html_tags) + s[pos:]
    return s


In [26]:
import random
# Create the disturbed dataset for drift analysis
valid_disturbed = valid[['review', 'is_positive']].copy()
print("valid_disturbed initialized.")

valid_disturbed initialized.


In [27]:
!pip install deep_translator sentencepiece sacremoses  # sentencepiece/sacremoses are for the local fallback model (transformers is already in Colab)
import random
from deep_translator import GoogleTranslator

import re
import os

translation_counts = {'google': 0, 'local_model': 0, 'failed': 0}
google_failures_in_a_row = 0

def translate_str(s):
    # 1st try: Google Translate (as in the course). Returns None if Google refuses.
    # After 3 failures in a row I stop asking Google, so I don't spam it.
    global google_failures_in_a_row
    if google_failures_in_a_row >= 3:
        return None
    try:
        translated = GoogleTranslator(source='auto', target='fr').translate(s)
        if translated:
            google_failures_in_a_row = 0
            return translated
    except Exception:
        pass
    google_failures_in_a_row += 1
    return None

def split_into_chunks(s, max_chars=400):
    # The model works best on short texts, so I cut a review into groups of sentences (max ~400 characters)
    chunks, current_chunk = [], ''
    for sentence in re.split(r'(?<=[.!?])\s+', s.strip()):
        if current_chunk and len(current_chunk) + len(sentence) > max_chars:
            chunks.append(current_chunk)
            current_chunk = sentence
        else:
            current_chunk = (current_chunk + ' ' + sentence).strip()
    return chunks + [current_chunk] if current_chunk else chunks

def translate_with_local_model(texts, batch_size=16):
    # Fallback: a local English -> French model from Hugging Face (MarianMT, Helsinki-NLP/opus-mt-en-fr).
    # The first time, it downloads the model (about 300 MB). I translate all the chunks together, in batches.
    import torch
    from transformers import MarianMTModel, MarianTokenizer
    torch.set_num_threads(min(4, os.cpu_count() or 1))   # more threads was slower on a shared CPU
    model_name = 'Helsinki-NLP/opus-mt-en-fr'
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).eval()

    all_chunks, owner = [], []
    for text_number, text in enumerate(texts):
        for chunk in split_into_chunks(text):
            all_chunks.append(chunk)
            owner.append(text_number)
    order = sorted(range(len(all_chunks)), key=lambda i: len(all_chunks[i]))   # similar lengths in one batch = faster
    translated_chunks = [None] * len(all_chunks)
    with torch.inference_mode():
        for start in range(0, len(order), batch_size):
            batch_ids = order[start:start + batch_size]
            batch = tokenizer([all_chunks[i] for i in batch_ids], return_tensors='pt',
                              padding=True, truncation=True, max_length=512)
            output_ids = model.generate(**batch, num_beams=1, no_repeat_ngram_size=4,
                                        max_new_tokens=int(batch['input_ids'].shape[1] * 1.5) + 10)
            for i, french in zip(batch_ids, tokenizer.batch_decode(output_ids, skip_special_tokens=True)):
                translated_chunks[i] = french
    results = [[] for _ in texts]
    for i, french in enumerate(translated_chunks):
        results[owner[i]].append(french)
    return [' '.join(parts) for parts in results]

def translate_many(reviews):
    # I translate a whole column: Google first, then the local model for every review Google did not translate.
    translated = {}
    for index, text in reviews.items():
        google_result = translate_str(text)
        if google_result is not None:
            translated[index] = google_result
            translation_counts['google'] += 1
    missing = [index for index in reviews.index if index not in translated]
    if missing:
        print(f"Google Translate did not translate {len(missing)} reviews -> I use the local model for them.")
        try:
            for index, french in zip(missing, translate_with_local_model([reviews[i] for i in missing])):
                translated[index] = french
                translation_counts['local_model'] += 1
        except Exception as error:
            print("Local model failed:", error)
            for index in missing:
                translated[index] = reviews[index]       # keep the English text
                translation_counts['failed'] += 1
    return pd.Series(translated)[reviews.index]

random_html_tags = (
    '<body>, </body>', '<html><body>', '</body></html>', '<h1>', '</h1>',
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>'
)

def inject_random_html_tags(s):
    num_tags = 25
    s_list = list(s)
    for i in range(num_tags):
        pos = random.randint(0, len(s_list))
        tag = random.choice(random_html_tags)
        s_list.insert(pos, tag)
    return ''.join(s_list)

valid_disturbed = valid[['review', 'is_positive']].copy()
disturbed_num = int(len(valid_disturbed) * 0.5)
random.seed(42)
disturbed_ind = random.sample(list(valid_disturbed.index), k=disturbed_num)

html_split = int(disturbed_num * 0.2)
valid_disturbed.loc[disturbed_ind[:html_split], 'review'] = \
    valid_disturbed.loc[disturbed_ind[:html_split], 'review'].apply(inject_random_html_tags)

valid_disturbed.loc[disturbed_ind[html_split:], 'review'] = \
    translate_many(valid_disturbed.loc[disturbed_ind[html_split:], 'review'])

print("Données perturbées avec succès (HTML + Traduction) !")
print(f"Reviews with HTML tags injected: {html_split}")
print(f"Reviews to translate into French: {disturbed_num - html_split}")
print(f"  translated by Google Translate: {translation_counts['google']}")
print(f"  translated by the local model (Helsinki-NLP/opus-mt-en-fr): {translation_counts['local_model']}")
print(f"  failed (kept in English): {translation_counts['failed']}")
print("Example of a translated review:", valid_disturbed.loc[disturbed_ind[html_split], 'review'][:300])

Google Translate did not translate 148 reviews -> I use the local model for them.


/workspace/w4d3/venv_ninja/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 47824.74it/s]

[transformers] Both `max_new_tokens` (=43) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=56) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (9 outputs left out)

[transformers] Both `max_new_tokens` (=194) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=193) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=287) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Données perturbées avec succès (HTML + Traduction) !
Reviews with HTML tags injected: 37
Reviews to translate into French: 148
  translated by Google Translate: 0
  translated by the local model (Helsinki-NLP/opus-mt-en-fr): 148
  failed (kept in English): 0
Example of a translated review: « J'ai eu des migraines en série et TOUS les autres types de maux de tête pendant de nombreuses années après un accident de voiture. Ils ont eu tellement mauvais I&#039,d commencer à crier de la douleur. Dans les urgences un médecin merveilleux m'a donné stadol spray nasal. En moins de dix minutes, 


In [28]:
# Use the trained pipeline to predict the probability of reviews being positive (class 1) for each entry in valid_disturbed
# The 'predict_proba' method returns an array of probabilities for both classes (0 and 1) for each review
# By indexing with [:,1], we select the probability that corresponds to class 1 (positive review)
# These probabilities are then assigned to a new column in valid_disturbed, named 'predict_proba'
valid_disturbed['predict_proba'] = pipeline.predict_proba(valid_disturbed['review'].values)[:,1]

In [29]:
column_mapping = ColumnMapping()
column_mapping.target = 'is_positive'
column_mapping.prediction = 'predict_proba'
column_mapping.text_features = ['review']
column_mapping.numerical_features = []
column_mapping.categorical_features = []

performance_report = Report(metrics=[ClassificationQualityMetric()])
# I compare the reference data with the DISTURBED validation data (the template had current_data=valid here,
# which would just repeat the first report).
performance_report.run(reference_data=reference, current_data=valid_disturbed, column_mapping=column_mapping)
performance_report

I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


Oops! Model accuracy has dropped. Let's look at the Data Drift report to see why

> **My note:** in my saved run, accuracy goes from **0.836** (clean validation data) to **0.741** on the disturbed data. Recall drops from 0.94 to 0.799 and ROC AUC from 0.728 to 0.698. In that run, 37 reviews got HTML tags and 148 reviews were translated into French: 0 by Google Translate (it blocked my computer) and 148 by the local Helsinki-NLP/opus-mt-en-fr model, with 0 failures.

In [30]:
import nltk

#Words List: This dataset contains a list of English words and is useful for various language processing tasks.
nltk.download('words')

#WordNet: A large lexical database of English, useful for finding the meanings of words, synonyms, antonyms, and more.
nltk.download('wordnet')

#Open Multilingual Wordnet (OMW): This dataset extends WordNet to multiple languages, allowing for cross-lingual lemmatization and analysis.
nltk.download('omw-1.4')

[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     SSRF attempt to restricted IP 198.18.0.1
[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.1
[nltk_data] Error loading omw-1.4: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.1


False

You will complete a function to generate a data drift report. Data drift refers to the change in the model input data distribution over time, which can significantly impact the model's performance. To monitor and detect data drift, you'll use a predefined Report class that can analyze various metrics indicative of data drift.

Complete the setup and execution of a data drift report using the Report class. This report will focus on monitoring the drift in specific columns of your dataset, including the target variable is_positive, the model's prediction probabilities predict_proba, and the textual content in the review column.

Instantiate the Report Class: Create an instance of the Report class, specifying the metrics you want to include for detecting data drift. You will track drift in:
- The is_positive column, which contains binary labels indicating negative or positive reviews.
- The predict_proba column, which contains the model's prediction probabilities.
- The textual content within the review column, using a metric that captures changes in text descriptors.


Hint: To monitor column-specific drift, use ColumnDriftMetric for numerical or categorical data and TextDescriptorsDriftMetric for textual data. Specify the column names directly in these metrics.

In [31]:
data_drift_report = Report(
    metrics=[
        ColumnDriftMetric('is_positive'),
        ColumnDriftMetric('predict_proba'),
        TextDescriptorsDriftMetric(column_name='review'),
    ]
)

data_drift_report.run(reference_data=reference, current_data=valid_disturbed, column_mapping=column_mapping)
data_drift_report

[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.1
[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     SSRF attempt to restricted IP 198.18.0.1


I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


Here we see the culprit: new (perturbed) dataset contains considerably more suspiciously long reviews and reviews with a lot of OOV (out-of-vocabulary) words

If we look at the examples of such reviews we see the problems right away:

*   HTML tags not being removed from the texts properly
*   Reviews in a new unexpected language

> **My note:** in my saved run, the text descriptors detect drift for all three: Text Length (K-S p = 0.0001), Non Letter Character % (p = 0.017) and OOV % (p ≈ 0.0). The prediction drift on `predict_proba` is also detected (p = 0.034), while the target `is_positive` is not (p = 0.625). The content drift check on `review` below also detects drift (score 0.697). The long reviews come from the injected HTML tags, and many OOV words come from the French reviews (they show up with an OOV share above 30% in the examples below).

In [32]:
from evidently.features.text_length_feature import TextLength
from evidently.features.OOV_words_percentage_feature import OOVWordsPercentage

In [33]:
# Corrected access to valid_disturbed
if 'valid_disturbed' in locals():
    text_feature = TextLength(column_name='review').generate_feature(data=valid_disturbed, data_definition=None)
    oov_feature = OOVWordsPercentage(column_name='review').generate_feature(data=valid_disturbed, data_definition=None)
    print("Features generated successfully.")
else:
    print("Error: valid_disturbed is not defined.")

Features generated successfully.


[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.1
[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     SSRF attempt to restricted IP 198.18.0.1


In [34]:
valid_disturbed['text_length'] = text_feature.values
valid_disturbed['oov_share'] = oov_feature.values

In [35]:
valid_disturbed[valid_disturbed['text_length'] > 1000].head()

,review,is_positive,predict_proba,text_length,oov_share
48578,"""I have b<h1>een on this medicine for 3 ye</h1...",1,0.0,1398,61.417323
21625,"""I have been on 4-5, 15 mg oxycodones a day fo...",1,1.0,4185,65.435356
26168,"""I <svg xmlns=""http://www.w3.org/2000/svg"" vie...",0,0.0,5264,88.043478
15764,"""I am 26 years old and <body>, </body>have a 4...",1,1.0,2354,66.019417
24116,"""W</h1>e<body>, </body>nt t<h1>o the hospital ...",1,0.0,4445,86.858974


In [36]:
valid_disturbed[valid_disturbed['text_length'] > 1000].iloc[0, 0]

'"I have b<h1>een on this medicine for 3 ye</h1></body></html>ars for pain relief for many prob</body></html>lems. I take 10mg 3 times</body></html> a day and I would s<body>, </body>uggest<h1> nothing else. I have be<body>, </body>en on other narco<html><body>ti<html><body>cs <html><body>th<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1">

In [37]:
valid_disturbed[valid_disturbed['oov_share'] > 30].head()

,review,is_positive,predict_proba,text_length,oov_share
19899,«Toute fois que j'ai de la douleur par surmena...,1,0.0,631,75.229358
1558,« Je prends dilaudid pour une grave condition ...,1,1.0,856,61.184211
48578,"""I have b<h1>een on this medicine for 3 ye</h1...",1,0.0,1398,61.417323
18574,« Pendant que j'étais dans la salle des urgenc...,1,1.0,360,62.068966
21625,"""I have been on 4-5, 15 mg oxycodones a day fo...",1,1.0,4185,65.435356


In [38]:
data_drift_dataset_report = Report(metrics=[
    ColumnDriftMetric(column_name='review')
])

data_drift_dataset_report.run(reference_data=reference,
                              current_data=valid_disturbed,
                              column_mapping=column_mapping)
data_drift_dataset_report

I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


# Optional : Content drift

Technical issues sorted out, the model continues to be used for reviews' sentiment analysis. Suppose we decide to apply it on reviews for antidepressants.

In [39]:
new_content = raw_data.loc[(raw_data['condition'] == 'Depression') & (raw_data['rating'].isin([1, 10])), ['review', 'rating']]
new_content['is_positive'] = new_content['rating'].apply(lambda x: 0 if x == 1 else 1)
new_content.drop(['rating'], inplace=True, axis=1)
new_content.head()

,review,is_positive
0,"""I&#039;ve tried a few antidepressants over th...",1
67,"""Zoloft did not help me at all. I was on it f...",0
73,"""Sadly only lasted 5 days on Effexor XR. The s...",0
81,"""I took Effexor for the first and last time ye...",0
135,"""Very good for GAD mixed with depression. I fe...",1


In [40]:
new_content['predict_proba'] = pipeline.predict_proba(new_content['review'].values)[:,1]

In [41]:
performance_report = Report(metrics=[
    ClassificationQualityMetric(),
])

performance_report.run(reference_data=valid, current_data=new_content,
                        column_mapping=column_mapping)
performance_report

I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


Unfortunately, model's performance is worse than expected. Let's look at the Data Drift report

> **My note:** accuracy on the depression reviews is **0.779**, against **0.836** on the pain validation data (ROC AUC 0.683 vs 0.728).

In [42]:
data_drift_report = Report(
    metrics=[
        ColumnDriftMetric('is_positive'),
        ColumnDriftMetric('predict_proba'),
        TextDescriptorsDriftMetric(column_name='review'),
    ]
)

data_drift_report.run(reference_data=reference,
                      current_data=new_content,
                      column_mapping=column_mapping)
data_drift_report

[nltk_data] Error loading wordnet: Security Violation
[nltk_data]     [pathsec.urlopen]: SSRF attempt to restricted IP
[nltk_data]     198.18.0.1
[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     SSRF attempt to restricted IP 198.18.0.1


I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.



You can see that there is drift to the data. Explain what is the problem exactly ?

**My answer (TODO):** The problem is **content drift**: the model is now used on a different kind of text than it was trained on. I trained it on reviews of **pain** medicines, and now I apply it to reviews of **antidepressants** (condition = Depression). The reviews are about different things and use different words, so the model sees many words it never learned.

What the Evidently reports show in my run:
- **Model quality dropped:** accuracy went from **0.836** (pain validation data) to **0.779** (depression reviews), and ROC AUC from **0.728** to **0.683**.
- **Target drift:** the share of positive reviews changed (about 77% in the pain training data, about 71% in the depression data). Z-test p-value = **0.002**, so drift is detected for `is_positive`.
- **Prediction drift:** the distribution of `predict_proba` changed too. K-S p-value = **0.002**, so drift is detected.
- **Text descriptors:** the review length (K-S p ≈ 0.0) and the share of out-of-vocabulary words (OOV %, p ≈ 0.00002) changed a lot. The share of non-letter characters did not change (p = 0.318), so this is not a technical problem like the HTML tags. It is the *content* that is new.

So the low p-values do not mean that something is broken in the pipeline. They mean that the new data comes from a different "world" (depression instead of pain), and the model's knowledge does not transfer well.

Let's see how Evidently can help to detect such a change

To detect content drift Evidently uses domain classifier approach. A classifier is trained that tries to predict whether a text is from a reference dataset or from a new dataset. If it can be done successfully than the new dataset is significantly different from the reference one.

If content data drift is detected Evidently also provides some insights on the nature of the drift:
* Words that are more distinctive of the current vs reference dataset.
These are the words that are the most informative for the domain classifier when it predicts if a text came from the reference or from the current dataset
* Examples of texts that are more distinctive of the current vs reference dataset. These examples were the easiest for a classifier to label correctly

In [43]:
data_drift_dataset_report = Report(metrics=[
    ColumnDriftMetric(column_name='review')
])

data_drift_dataset_report.run(reference_data=reference,
                              current_data=new_content,
                              column_mapping=column_mapping)
data_drift_dataset_report


I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


At once we can see how the current dataset differs from the reference dataset our model was trained on. Current dataset is characterized with words and examples about depression, mood and popular antidepressants while the reference dataset is more about pain, shock symptoms and popular painkillers

> **My note:** in my run the content drift score is **0.94** and drift is detected. The most typical words of the new data are *depression, antidepressant, anxiety, prozac, pristiq, zoloft*, and those of the reference are *pain, migraine, painkiller, oxycontin, arthritis, relief*.

Note that no such drift is detected for validation dataset that consists of reviews for painkillers, similar to reference dataset

> **My note:** for the pain validation data the drift score is only **0.474**, so no drift is detected, as expected.

In [44]:
data_drift_dataset_report = Report(metrics=[
    ColumnDriftMetric(column_name='review')
])

data_drift_dataset_report.run(reference_data=reference,
                              current_data=valid,
                              column_mapping=column_mapping)
data_drift_dataset_report

I generated the Evidently interactive report when I ran this cell. I did not save the large HTML in this notebook so the GitHub file stays small. Re-run this cell in Colab to see the interactive report. The key results are written in my notes in the markdown cells nearby.


What could be one of the solution to deal with this kind of data change ?

> Ajouter une citation



**My answer (TODO):** The main solution is to **retrain the model on data that looks like the new data**. I would collect and label some antidepressant reviews (rating 1 → negative, rating 10 → positive) and retrain the TF-IDF + SGDClassifier pipeline on pain + depression reviews together, or train a separate model for each condition. This is often called *continuous retraining* or *adaptive learning*: I keep updating the model with new labelled data so it follows the changes in the data.

To make this safe in production, I would also:
- keep **monitoring** with Evidently: the text content drift report (domain classifier) and the text descriptors (length, OOV %) can warn me *before* I even have labels, and the classification report confirms the drop once labels arrive;
- set an **alert threshold** (for example, retrain when content drift is detected or accuracy goes below 0.80);
- until the new model is ready, **only use the model for the domain it was trained on** (pain reviews), or flag predictions on new conditions as less reliable.

### Solution : Mise en place d'un Pipeline Robuste

Pour corriger le drift causé par les tags HTML, nous pouvons créer un transformateur personnalisé qui nettoie le texte avant qu'il ne soit envoyé au `TfidfVectorizer`.

In [45]:
import re
from sklearn.base import BaseEstimator, TransformerMixin

class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Cette fonction retire les tags HTML à l'aide d'une expression régulière
        clean_re = re.compile('<.*?>')
        if isinstance(X, pd.Series):
            return X.apply(lambda x: re.sub(clean_re, '', str(x)))
        return [re.sub(clean_re, '', str(text)) for text in X]

# Nouveau pipeline incluant le nettoyage
robust_pipeline = Pipeline([
    ('cleaner', TextCleaner()),
    ('tfidf', TfidfVectorizer(sublinear_tf=True, max_df=0.5, stop_words='english')),
    ('classifier', SGDClassifier(alpha=0.0001, max_iter=50, penalty='l1', loss='modified_huber', random_state=42))
])

robust_pipeline.fit(X_train, y_train)
print("Pipeline robuste entraîné ! ✅")

Pipeline robuste entraîné ! ✅


In [46]:
from sklearn.metrics import accuracy_score

# Mise à jour des probabilités pour le dataset perturbé
valid_disturbed['predict_proba'] = robust_pipeline.predict_proba(valid_disturbed['review'].values)[:,1]

# Prédictions avec le pipeline original (sensible au drift)
y_pred_orig = pipeline.predict(valid_disturbed['review'])
acc_orig = accuracy_score(valid_disturbed['is_positive'], y_pred_orig)

# Prédictions avec le pipeline robuste (nettoyage HTML inclus)
y_pred_robust = robust_pipeline.predict(valid_disturbed['review'])
acc_robust = accuracy_score(valid_disturbed['is_positive'], y_pred_robust)

print(f"Précision du pipeline original sur données bruitées : {acc_orig:.2%}")
print(f"Précision du pipeline robuste sur données bruitées  : {acc_robust:.2%}")

if acc_robust > acc_orig:
    print("\nSuccès ! Le pipeline robuste est plus performant grâce au nettoyage des tags HTML. ✅")
else:
    print("\nLa précision est similaire car la traduction française reste le défi majeur pour ce modèle.")

Précision du pipeline original sur données bruitées : 74.12%
Précision du pipeline robuste sur données bruitées  : 75.74%

Succès ! Le pipeline robuste est plus performant grâce au nettoyage des tags HTML. ✅
